# 12 - External Evaluation L78 Hybrid

Executa benchmarks externos no Colab com `lm-evaluation-harness`, comparando `student_pruned_base` contra `l78_adapter`.

In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, math

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    !pip -q install transformers accelerate sentencepiece safetensors peft datasets
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
UTILS_DIR = PROJECT_ROOT / 'utils'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, UTILS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)

OUT_DIR = RESULTS_DIR / 'external_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)


In [ ]:
# Instalação opcional. Rode se for executar benchmarks externos nesta sessão.
!pip -q install lm-eval accelerate sentencepiece peft transformers safetensors


In [ ]:
# Configuração recomendada inicial.
# lm-eval deve carregar o modelo base e aplicar o adapter PEFT separadamente.
# Isto avalia qualidade do modelo funcional, não velocidade/runtime packed.
BASE_MODEL = CHECKPOINTS_DIR / 'student_pruned'
L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'

if not BASE_MODEL.exists():
    raise FileNotFoundError(f'Base model não encontrado: {BASE_MODEL}')
if not L78_ADAPTER.exists():
    raise FileNotFoundError(f'Adapter l78 não encontrado: {L78_ADAPTER}')

MODEL_ARGS_L78 = f'pretrained={BASE_MODEL},peft={L78_ADAPTER},trust_remote_code=True'
MODEL_ARGS_BASE = f'pretrained={BASE_MODEL},trust_remote_code=True'

TASKS_FAST = ['arc_easy', 'hellaswag']
TASKS_FULL = ['arc_easy', 'arc_challenge', 'hellaswag', 'truthfulqa_mc1']
BATCH_SIZE = 1
LIMIT = 100  # use None para full benchmark

commands = []
for model_label, model_args in [('student_pruned_base', MODEL_ARGS_BASE), ('l78_adapter', MODEL_ARGS_L78)]:
    for tasks, eval_label in [(TASKS_FAST, 'fast'), (TASKS_FULL, 'full')]:
        task_str = ','.join(tasks)
        limit_part = f' --limit {LIMIT}' if LIMIT else ''
        out_file = OUT_DIR / f'{model_label}_{eval_label}.json'
        cmd = f'lm_eval --model hf --model_args "{model_args}" --tasks {task_str} --batch_size {BATCH_SIZE}{limit_part} --output_path {out_file}'
        commands.append({'model_label': model_label, 'eval_label': eval_label, 'tasks': tasks, 'command': cmd, 'output_path': str(out_file)})

print(json.dumps(commands, ensure_ascii=False, indent=2))


In [ ]:
plan = {
    'models': {
        'student_pruned_base': str(BASE_MODEL),
        'l78_adapter': str(L78_ADAPTER),
    },
    'model_args': {
        'student_pruned_base': MODEL_ARGS_BASE,
        'l78_adapter': MODEL_ARGS_L78,
    },
    'commands': commands,
    'interpretation': {
        'purpose': 'Check whether local 0.92 recovery generalizes beyond the 100-question suite.',
        'compare_against': ['student_pruned_base', 'l78_adapter'],
        'warning': 'This evaluates model quality through HF/PEFT loading. It does not benchmark packed ternary runtime speed or execute mlp_ternary_2bit.bin.',
        'next_step_if_good': 'Build custom loader/runtime path that uses manifest.json + mlp_ternary_2bit.bin for MLP execution.'
    }
}
plan_path = OUT_DIR / 'external_eval_plan.json'
plan_path.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', plan_path)


## Execução

Rode primeiro as células `fast`. Se funcionarem, rode as células `full`. Os JSONs serão salvos em `results/external_eval/` no Drive.

In [ ]:
# Executa FAST baseline.
cmd = next(c['command'] for c in commands if c['model_label'] == 'student_pruned_base' and c['eval_label'] == 'fast')
print(cmd)
!{cmd}


In [ ]:
# Executa FAST l78 adapter.
cmd = next(c['command'] for c in commands if c['model_label'] == 'l78_adapter' and c['eval_label'] == 'fast')
print(cmd)
!{cmd}


In [ ]:
# Resume resultados FAST se os arquivos existirem.
def read_result(path):
    path = Path(path)
    if not path.exists():
        return {'exists': False, 'path': str(path)}
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return {'exists': True, 'path': str(path), 'keys': list(data.keys()), 'results': data.get('results', data)}

fast_summary = {}
for c in commands:
    if c['eval_label'] == 'fast':
        fast_summary[c['model_label']] = read_result(c['output_path'])

fast_summary_path = OUT_DIR / 'fast_summary.json'
fast_summary_path.write_text(json.dumps(fast_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(fast_summary, ensure_ascii=False, indent=2)[:8000])
print('saved', fast_summary_path)


## Full Benchmarks

Rode estas células só se os dois comandos `fast` funcionarem.

In [ ]:
# Executa FULL baseline limitado.
cmd = next(c['command'] for c in commands if c['model_label'] == 'student_pruned_base' and c['eval_label'] == 'full')
print(cmd)
!{cmd}


In [ ]:
# Executa FULL l78 adapter limitado.
cmd = next(c['command'] for c in commands if c['model_label'] == 'l78_adapter' and c['eval_label'] == 'full')
print(cmd)
!{cmd}


In [ ]:
# Resume todos os resultados disponíveis.
all_summary = {}
for c in commands:
    all_summary[f"{c['model_label']}_{c['eval_label']}"] = read_result(c['output_path'])

summary_path = OUT_DIR / 'external_eval_summary.json'
summary_path.write_text(json.dumps(all_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(all_summary, ensure_ascii=False, indent=2)[:12000])
print('saved', summary_path)
